# Hankel Transform of a Tabulated Form Factor (Python)

This notebook shows how to use LibHankel when the form factor is not available as an analytic expression but as a table of $(q, F(q))$ values.

We use the sphere form factor as a concrete example — sampling it on a coarse $q$ grid and passing those values as tabulated input — then compare different strategies for runtime and accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import libhankel

## Set up parameters and evaluation points

In [ ]:
nu = 0
params = np.array([10.0, 1.0])   # sphere: [radius R, contrast eta]

r_arr = np.linspace(1.0, 25.0, 50)

## Build the tabulated form factor

The sphere form factor in LibHankel is the intensity $|F(q)|^2$:

$$F(q) = \left( \eta \cdot 4\pi \frac{\sin(qR) - qR\cos(qR)}{q^3} \right)^2$$

We evaluate this at a coarse logarithmic grid of $q$ values to produce the tabulated data.

In [ ]:
R, eta = params[0], params[1]

q_data = np.logspace(-2, 1, 40)   # 40 points from q=0.01 to q=10
qR = q_data * R

# Use series expansion for small qR to avoid numerical cancellation
f_data = np.where(
    qR < 1e-4,
    (eta * (4.0/3.0) * np.pi * R**3 * (1 - qR**2/10 + qR**4/280))**2,
    (eta * 4.0 * np.pi * (np.sin(qR) - qR * np.cos(qR)) / q_data**3)**2,
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(q_data, f_data)
ax.set_xlabel("q")
ax.set_ylabel("F(q)")
ax.set_title("Tabulated sphere form factor (intensity)")
plt.tight_layout()
plt.show()

## Compute the Hankel transform with the tabulated form factor

The tabulated form factor is passed as a `dict` with keys:
- `q`: the scattering vector values
- `f`: the corresponding form factor values
- `interp_type`: interpolation method (`"linear"` or `"cubic"`)
- `tail`: behaviour outside the tabulated range (`"zero"` or `"power_law"`)

Here is a basic example using FFTLog:

In [ ]:
tabulated_ff = {
    "q": q_data,
    "f": f_data,
    "interp_type": "cubic",
    "tail": "power_law",
}

result = libhankel.hankel_transform(nu, tabulated_ff, r_arr, [], "FFTLog", {})
print("G(r):", result)

## Comparing strategies

We compare four strategies on the cubic-interpolated tabulated form factor:

- **FFTLog** — O(N log N) transform, no convergence parameters
- **DHT_Key_201** — fixed digital filter grid, fast and non-adaptive
- **QWE_Key** — adaptive quadrature with convergence control
- **Adaptive_DE_Ooura** — adaptive double-exponential quadrature

### G(r) across strategies

We compute the Hankel transform using four strategies and overlay the results to confirm they agree.

In [ ]:
tab_ff = {"q": q_data, "f": f_data, "interp_type": "cubic", "tail": "power_law"}
strat_params = {"n_eval": 250, "eps_rel": 1e-9}

strategies = {
    "FFTLog":             {},
    "DHT_Key_201":        strat_params,
    "QWE_Key":            strat_params,
    "Adaptive_DE_Ooura":  strat_params,
}

fig, ax = plt.subplots(figsize=(7, 4))

for name, p in strategies.items():
    Gr = np.array(libhankel.hankel_transform(nu, tab_ff, r_arr, [], name, p))
    ax.plot(r_arr, Gr, label=name)

ax.set_xlabel("r")
ax.set_ylabel("G(r)")
ax.set_title("G(r) — tabulated sphere form factor, all strategies")
ax.legend()
plt.tight_layout()
plt.show()

### Runtime comparison

In [ ]:
import timeit

tab_ff_cubic = {"q": q_data, "f": f_data, "interp_type": "cubic", "tail": "power_law"}
adaptive_params = {"n_eval": 250, "eps_rel": 1e-9}
n_repeats = 10

def time_strategy(strategy, params):
    return timeit.timeit(
        lambda: libhankel.hankel_transform(nu, tab_ff_cubic, r_arr, [], strategy, params),
        number=n_repeats,
    ) / n_repeats

t_fftlog   = time_strategy("FFTLog",           {})
t_dht      = time_strategy("DHT_Key_201",      adaptive_params)
t_qwe      = time_strategy("QWE_Key",          adaptive_params)
t_de_ooura = time_strategy("Adaptive_DE_Ooura", adaptive_params)

fig, ax = plt.subplots(figsize=(7, 4))

strategies = ["FFTLog", "DHT_Key_201\n(fixed grid)", "QWE_Key\n(adaptive)", "Adaptive_DE_Ooura\n(adaptive)"]
times_ms   = [t_fftlog * 1e3, t_dht * 1e3, t_qwe * 1e3, t_de_ooura * 1e3]

bars = ax.bar(strategies, times_ms)
ax.bar_label(bars, fmt="{:.2f} ms", padding=4)
ax.set_ylim(0, max(times_ms) * 1.3)
ax.set_ylabel("Mean runtime (ms)")
ax.set_title(f"Runtime comparison (n={len(r_arr)} points, averaged over {n_repeats} runs)")

plt.tight_layout()
plt.show()

### Accuracy comparison

We compare each strategy against the closed-form analytical solution for the sphere SESANS correlation function. Because `hankel_transform` does not include the $1/(2\pi)$ normalisation of the analytical formula, we compute $[G(r) - G(0)] / 2\pi$ from each numerical result before comparing, where $G(0)$ is evaluated via `hankel_G0` on the same tabulated form factor.

In [ ]:
def analytical_sphere(r_arr, R, eta):
    xi = r_arr / (2.0 * R)
    eps = 1e-15
    pref = eta**2 * np.pi * R**4
    G = np.zeros_like(r_arr, dtype=float)
    last_val = 0.0
    have_last = False
    for i, x in enumerate(xi):
        if 0.0 <= x < 1.0:
            xi2 = x**2
            sqrt_term = np.sqrt(1.0 - xi2)
            term1 = sqrt_term * (2.0 + xi2)
            log_arg = max(x / (1.0 + sqrt_term), eps)
            term2 = xi2 * (4.0 - xi2) * np.log(log_arg)
            G[i] = pref * (term1 + term2 - 2.0)
            last_val = G[i]
            have_last = True
    if have_last:
        for i, x in enumerate(xi):
            if not (0.0 <= x < 1.0):
                G[i] = last_val
    return G

G_analytical = analytical_sphere(r_arr, R, eta)
G0_tab = libhankel.hankel_G0(tab_ff_cubic, [], 1e-9)
scale = np.max(np.abs(G_analytical))

strategy_keys = ["FFTLog", "DHT_Key_201", "QWE_Key", "Adaptive_DE_Ooura"]

print(f"{'Strategy':<22} {'Max abs error':>16} {'Max rel error':>16}")
print("-" * 56)
for key in strategy_keys:
    params_s = {} if key == "FFTLog" else adaptive_params
    Gr = np.array(libhankel.hankel_transform(nu, tab_ff_cubic, r_arr, [], key, params_s))
    shifted = (Gr - G0_tab) / (2 * np.pi)
    abs_err = np.max(np.abs(shifted - G_analytical))
    rel_err = abs_err / scale
    print(f"{key:<22} {abs_err:>16.3e} {rel_err:>16.3e}")